In [1]:
from google.colab import drive
import os

# 1. Driveをマウント
drive.mount('/content/drive')

# 2. ログ保存用のフォルダを作成
LOG_DIR = "/content/drive/MyDrive/Research_Logs"
os.makedirs(LOG_DIR, exist_ok=True)

print(f"✅ ログ保存先: {LOG_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ログ保存先: /content/drive/MyDrive/Research_Logs


In [2]:
%pip install rank_bm25
from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

from DynamicScalingLlamaAttention import DynamicScalingLlamaAttention
from ScalingController import ScalingController
from get_model_safe import get_model_safe, test_patched_model_sanity, remove_cache
from main_inference import main_inference
from run_eval import run_eval_exact_match
from analyze_log import ExperimentAnalyzer


if 'CACHED_MODEL' not in globals():
    CACHED_MODEL = None
    CACHED_TOKENIZER = None

# attention テストコード
# test_patched_model_sanity("meta-llama/Llama-2-7b-chat-hf")

# hotpot datasetのロード
print("Loading dataset...")
dataset = load_dataset("hotpot_qa", "distractor", split="validation")

Loading dataset...


In [ ]:
# run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=1000, beta=1.0, scaling_method="log")
# run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=1000, beta=1.0, scaling_method="baseline")
# run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=1000, beta=1.0, scaling_method="existing")
# run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=1000, beta=1.0, scaling_method="probabilistic")
run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=1000, beta=1.0, scaling_method="distribution")

📝 Logging to: /content/drive/MyDrive/Research_Logs/experiment_hotpot_1.0_distribution_20251231_122155.jsonl
⏳ Loading new model... (This may take a while)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model loaded and patched successfully.
🚀 Exact Match Evaluation Start: Processing 1000 samples
------------------------------------------------------------
Applying scaling: distribution
Q1: Were Scott Derrickson and Ed Wood of the same nationality?
Gen: No, Scott Derrickson is American, while Ed Wood was American.
Ref: yes
Result: ❌ (EM: 0)
------------------------------
Applying scaling: distribution
Q2: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gen: Shirley Temple held the position of Secretary of State for Constitutional Affairs.
Ref: Chief of Protocol
Result: ❌ (EM: 0)
------------------------------
Applying scaling: distribution
Q3: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gen: Animorphs.
Ref: Animorphs
Result: ✅ (EM: 1)
------------------------------
Applying scaling: distribution
Q4: Are the Laleli Mosque a

KeyboardInterrupt: 

In [ ]:
dataset = load_dataset("hotpot_qa", "distractor", split="train")

run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=500, beta=1.0, scaling_method="probabilistic")
run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=500, beta=2.0, scaling_method="probabilistic")
run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=500, beta=3.0, scaling_method="probabilistic")
run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=500, beta=5.0, scaling_method="probabilistic")
run_eval_exact_match(dataset, "meta-llama/Llama-2-7b-chat-hf", num_samples=500, beta=10.0, scaling_method="probabilistic")

ストリーミング出力は最後の 5000 行に切り捨てられました。
------------------------------
Applying scaling: probabilistic
Q195: Which is currently more valuable, Temagami-Lorrain Mine or Meadowbank Gold Mine?
Gen: Based on the information provided in the documents, Meadowbank Gold Mine is currently more valuable than Temagami-Lorrain Mine. The Meadowbank Gold Mine is an open-pit gold mine that has been in operation since 2010 and has a proven and probable reserve of 2.3 million ounces of gold, with a gold grade of 0.65 grams per tonne. In contrast, the Temagami-Lorrain
Ref: Meadowbank Gold Mine
Result: ✅ (EM: 1)
------------------------------
Applying scaling: probabilistic
Q196: St James Street appears as a segment of Whitecross Street on the 1610 map of the Monmouth by an English historian best known as a mapmaker of what perior?
Gen: The answer is John Speed.
Ref: Stuart period
Result: ❌ (EM: 0)
------------------------------
Applying scaling: probabilistic
Q197: The Pineground Bridge formerly carried Depot R

In [ ]:
# 1. アナライザーの初期化
analyzer = ExperimentAnalyzer(log_dir="/content/drive/MyDrive/Research_Logs")

# 2. 特定のファイルのスコアだけ見たい場合
# res = analyzer.summarize_file("experiment_hotpot_1.0_distribution_20251224.jsonl")
# print(f"Average EM: {res['avg_em']}%")

# 3. フォルダ内の全結果をまとめて比較したい場合（推奨）
analyzer.report_all_logs()


📊 Experiment Summary Report
+-----------------------------------------------------------+----------+-----------+--------+--------------+
| File Name                                                 | Avg EM   |   Samples |   Beta | Method       |
+===========================================================+==========+===========+========+==============+
| experiment_hotpot_1.0_baseline_20251224_080759.jsonl      | 61.80%   |      1000 |      1 | distribution |
+-----------------------------------------------------------+----------+-----------+--------+--------------+
| experiment_hotpot_1.0_distribution_20251225_053936.jsonl  | 61.30%   |      1000 |      1 | distribution |
+-----------------------------------------------------------+----------+-----------+--------+--------------+
| experiment_hotpot_1.0_existing_20251224_071838.jsonl      | 60.00%   |       100 |      1 | distribution |
+-----------------------------------------------------------+----------+-----------+--------+------

In [ ]:
import json

def print_type_dataset(dataset):
  print(dataset)
  print(dict(dataset))

  # train データを確認
  print(dataset["train"][0].keys())
  # => 'id', 'question', 'context', 'answer'

  print("*"*100)
  print("trainの各keyの構造")
  print("question: ")
  print(dataset["train"][0]["question"])
  print("context: ")
  print(dataset["train"][0]["context"])
  print("answer: ")
  print(dataset["train"][0]["answer"])
  print("*"*100)

  # for item in dataset["train"][0]["context"]["title"]:
  #     print(item)

  print(dataset["train"][0]["context"].keys())

  print("*"*100)
  print("contextの各keyの構造")
  print("sentences: ")
  print("".join(dataset["train"][0]["context"]["sentences"][0]))

# print_type_dataset(dataset)
# print(dataset)
print(json.dumps(dataset[0], indent=2))

{
  "id": "5a8b57f25542995d1e6f1371",
  "question": "Were Scott Derrickson and Ed Wood of the same nationality?",
  "answer": "yes",
  "type": "comparison",
  "level": "hard",
  "supporting_facts": {
    "title": [
      "Scott Derrickson",
      "Ed Wood"
    ],
    "sent_id": [
      0,
      0
    ]
  },
  "context": {
    "title": [
      "Ed Wood (film)",
      "Scott Derrickson",
      "Woodson, Arkansas",
      "Tyler Bates",
      "Ed Wood",
      "Deliver Us from Evil (2014 film)",
      "Adam Collis",
      "Sinister (film)",
      "Conrad Brooks",
      "Doctor Strange (2016 film)"
    ],
    "sentences": [
      [
        "Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.",
        " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.",
        " Sarah Jessica Parker, Pa

In [ ]:
%pip install xopen
import json
from xopen import xopen
from datetime import datetime

class LitMDataLoader:
    """
    Lost in the Middle 形式の (.jsonl.gz) データセットを読み込むためのクラス
    """
    def __init__(self, base_dir="./qa_data"):
        self.base_dir = base_dir

    def get_file_path(self, num_docs, gold_at):
        # フォルダ構造: qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz
        return "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"

    def load_samples(self, num_docs, gold_at):
        file_path = self.get_file_path(num_docs, gold_at)
        if not os.path.exists(file_path):
            print(f"⚠️ File not found: {file_path}")
            return []

        samples = []
        with xopen(file_path, "r") as f:
            for line in f:
                samples.append(json.loads(line))
        return samples

def run_litm_eval(model_path, scaling_method, beta, num_docs=20, num_samples=50):
    loader = LitMDataLoader(base_dir="./qa_data")

    # モデルの準備 (既存の関数を使用)
    model, tokenizer = get_model_safe(model_path)
    gold_indices = [0]

    for gold_at in gold_indices:
        print(f"\n===== Testing: docs={num_docs} =====")

        # データのロード
        samples = loader.load_samples(num_docs, gold_at)
        if not samples: continue

        # サンプル数を制限
        samples = samples[:num_samples]

        # 実験ログの設定 (既存のExperimentLoggerを使用)
        exp_name = f"litm_d{num_docs}"
        logger = ExperimentLogger(exp_name=exp_name, dataset="nq_open")

        total_em = 0
        count = 0

        for i, item in enumerate(samples):
            query = item["question"]
            gold_answers = item["answers"] # LitM(NQ)はリスト形式

            # --- 文書の整形 (BM25は行わず、データセットの順序を維持) ---
            # LitMデータの item["ctxs"] は [{'title':..., 'text':...}, ...] の形式
            full_doc = [f"{ctx['title']}\n{ctx['text']}" for ctx in item["ctxs"]]

            tokenized_docs = []
            for document in full_doc:
                tokenized_docs.append(document.split())

            # BM25スコア計算
            bm25 = BM25Okapi(tokenized_docs)
            raw_scores = [float(s) for s in bm25.get_scores(query.split())]

            # スコア順に並べ替え (降順)
            combined = sorted(zip(raw_scores, full_doc), key=lambda x: x[0], reverse=True)
            sorted_scores, sorted_documents = zip(*combined)
            sorted_scores = list(sorted_scores)
            sorted_documents = list(sorted_documents)

            # print("="*50)
            # print("query: ", query)
            # print("="*50)
            # print("documents: ")
            # for document in full_doc:
            #   print(document)
            # print("="*50)
            # print("gold_answers: ")
            # print(gold_answers)
            # print("="*50)

            print("="*50)
            # --- 推論実行 (既存のメイン関数を使用) ---
            generated_answer1 = main_inference(
                model=model,
                tokenizer=tokenizer,
                documents=full_doc,
                scores=sorted_scores,
                query=query,
                scaling_method=scaling_method, # 任意
                beta=beta # 任意
            )
            print(generated_answer1)

            # --- 評価 (自作の calculate_substring_score を使用) ---
            em_score = calculate_substring_score(generated_answer1, gold_answers)
            total_em += em_score
            count += 1

            # ログ記録 (既存のロジック)
            logger.log_sample({
                "id": i,
                "gold_pos": gold_at,
                "em_score": em_score,
                "generated_answer": generated_answer1
            })

        print(f"📊 EM: {100.0 * total_em / count:.2f}%")


SyntaxError: parameter without a default follows parameter with a default (ipython-input-2303296485.py, line 29)

In [ ]:
model_path = "meta-llama/Llama-2-7b-chat-hf"

run_litm_eval(model_path=model_path, num_docs=20, num_samples=50, scaling_method="baseline", beta=1.0)
run_litm_eval(model_path=model_path, num_docs=20, num_samples=50, scaling_method="log", beta=1.0)
run_litm_eval(model_path=model_path, num_docs=20, num_samples=50, scaling_method="probabilistic", beta=1.0)
run_litm_eval(model_path=model_path, num_docs=20, num_samples=50, scaling_method="existing", beta=1.0)

In [10]:
# kv_retrieval task
%pip install xopen

# 設定
KV_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/KV/kv-retrieval-300_keys.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/kv_retrieval/test_action_kv_retrieval_strong_scale.jsonl"

SAMPLE_NUM = 50
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

IS_SCALE = True

# 構成は以下
# 1. prompt作成コード:format_kv_prompt 2. 処理コード:run_kv_retrieval 3. 評価コード:eval_kv 4. 全体の実行コード:main
# 最大の障害が、gold トークンに対してのスケーリング　これを実装するために、promptから対応トークンを割り出すことや、run_kv_retrievalへ渡す必要がある。

from xopen import xopen
import json
import gc
import torch
from tqdm import tqdm
import random

random.seed(0)

def format_kv_prompt(records, key)->str:
    """keyとrecordsからプロンプトの作成"""
    kv_dict = {r[0]: r[1] for r in records}
    context = json.dumps(kv_dict)

    header = "Extract the value corresponding to the specified key in the JSON object below.\n\nJSON data:\n"
    footer = f'\n\nKey: "{key}"\nCorresponding value:'

    prompt = header + context
    context_end_idx = len(prompt)
    prompt = prompt + footer

    return prompt, context_end_idx

def shuffle_records(records, sample_num, target_key, target_value):
    """
    recordsから正解ペアを確保し、他のレコードと混ぜてから、
    指定されたsample_numの範囲内のランダムな位置に正解を挿入する。
    """
    gold_pair = [target_key, target_value]
    other_records = [r for r in records if r != gold_pair]
    random.shuffle(other_records)
    selected_others = other_records[:sample_num - 1]
    random_pos = random.randint(0, len(selected_others))

    selected_others.insert(random_pos, gold_pair)

    return selected_others

def run_kv_retrieval(records, key, model, tokenizer):
  prompt, context_end_idx = format_kv_prompt(records, key)

  inputs = tokenizer(prompt, return_tensors="pt", return_offsets_mapping=True).to(model.device)
  input_ids = inputs.input_ids
  offsets = inputs.offset_mapping[0]
  seq_len = input_ids.shape[1]

  char_start = prompt.find(key, 0, context_end_idx)
  char_end = char_start + len(key)

  scale_map = torch.full((1, seq_len), 1.0, device=model.device, dtype=torch.float16)

  if char_start != -1:
    for t_i, (off_start, off_end) in enumerate(offsets):
      if off_start == off_end: continue
      if off_end > char_start and off_start < char_end:
        if IS_SCALE:
          scale_map[0, t_i] -= 10
        break
  else:
    return "⚠️ Gold key not found in the JSON section of the prompt."

  model.config.current_scale_map = scale_map

  with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=50, do_sample=False)

  output_text = tokenizer.decode(output[0][seq_len:], skip_special_tokens=True)
  return output_text.strip()


def main():
  model, tokenizer = get_model_safe(MODEL_NAME)

  with xopen(KV_PATH, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
    for line in tqdm(fin, desc="Running KV Retrieval..."):
      data = json.loads(line)
      key = data["key"]
      value = data["value"]
      records = data["ordered_kv_records"]
      records = shuffle_records(records, SAMPLE_NUM, key, value)

      try:
        output = run_kv_retrieval(records, key, model, tokenizer)
      except Exception as e:
        print(f"Exception occured at {key}, {value}:{e}")
        continue

      result_record = {
          "output": output,
          "key": key,
          "value": value,
          "num_kv": len(records),
          "target_index": next((i for i, r in enumerate(records) if r[0]==key), -1),
          "records": records,
      }

      fout.write(json.dumps(result_record)+"\n")
      fout.flush()

  # print("Cleaning Cache")
  # del model, tokenizer
  # gc.collect()
  # torch.cuda.empty_cache()

if __name__ == "__main__":
  main()

✅ Model already loaded. Using cached model.


Running KV Retrieval...: 2it [00:06,  3.10s/it]


KeyboardInterrupt: 

In [ ]:
# 終了用コード
from google.colab import runtime

runtime.unassign()

In [5]:
# kv_retrieval task
%pip install xopen

from xopen import xopen
import json
import gc
import torch
import torch.nn as nn
from tqdm import tqdm
import random

random.seed(0)

def calculate_nll(model, tokenizer, prompt, value, kv_positions):
    """model()によるprefillを行い、ppl, attention mapを返す"""
    full_text = prompt + " " + value
    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)

    kv_token_positions = []
    for start_char, end_char in kv_positions:
        start_idx = inputs.char_to_token(start_char)
        end_idx = inputs.char_to_token(end_char-1)
        kv_token_positions.append((start_idx, end_idx))

    input_ids = inputs.input_ids

    target_ids = tokenizer(value, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    target_len = target_ids.shape[1]

    with torch.no_grad():
        outputs = model(input_ids)

    logits = outputs.logits # [1, seq_len, vocab_size]
    target_logits = logits[0, -(target_len+1):-1, :]
    target_labels = input_ids[0, -target_len:]

    # 15～20層の最後のトークンに対するアテンションマップを取得
    target_layers_attentions = outputs.attentions[14:20] # [num_layers, batch_size, num_heads, seq_len, seq_len]
    last_token_attentions = torch.stack([layer[:,:,-1,:] for layer in target_layers_attentions]) # (target_layers, bsz, num_heads, 1, seq_len)

    # 負の対数尤度の平均
    loss_fct = nn.CrossEntropyLoss(reduction="mean")
    avg_nll = loss_fct(target_logits, target_labels)

    del outputs

    return {
        "avg_nll": avg_nll.item(),
        "target_len": target_len,
        "kv_token_positions": kv_token_positions,
    }, last_token_attentions

def format_kv_prompt(records, key)->str:
    """keyとrecordsからプロンプトの作成"""
    kv_dict = {r[0]: r[1] for r in records}
    context = json.dumps(kv_dict)

    header = "Extract the value corresponding to the specified key in the JSON object below.\n\nJSON data:\n"
    footer = f'\n\nKey: "{key}"\nCorresponding value:'

    prompt = header + "{"
    positions = []
    for i, record in enumerate(records):
      kv_str = json.dumps({record[0]: record[1]})[1:-1]

      start_idx = len(prompt)
      prompt += kv_str
      end_idx = len(prompt)

      positions.append((start_idx, end_idx))
      if i < len(records) -1:
        prompt += ", "
    prompt += "}"
    context_end_idx = len(prompt)
    prompt = prompt + footer

    return prompt, context_end_idx, positions

def shuffle_records(records, sample_num, target_key, target_value):
    """
    recordsから正解ペアを確保し、他のレコードと混ぜてから、
    指定されたsample_numの範囲内のランダムな位置に正解を挿入する。
    """
    gold_pair = [target_key, target_value]
    other_records = [r for r in records if r != gold_pair]
    random.shuffle(other_records)
    selected_others = other_records[:sample_num - 1]
    random_pos = random.randint(0, len(selected_others))

    selected_others.insert(random_pos, gold_pair)

    return selected_others

def run_kv_retrieval(records, key, value, model, tokenizer):
  # プロンプトの作成
  prompt, context_end_idx, kv_char_positions = format_kv_prompt(records, key)
  full_text = prompt + " " + value

  # トークナイズ関連
  inputs = tokenizer(full_text, return_tensors="pt", return_offsets_mapping=True).to(model.device)
  input_ids = inputs.input_ids
  offsets = inputs.offset_mapping[0]
  seq_len = input_ids.shape[1]

  char_start = prompt.find(key, 0, context_end_idx)
  char_end = char_start + len(key)

  scale_map = torch.full((1, seq_len), 1.0, device=model.device, dtype=torch.float16)

  if char_start != -1:
    for t_i, (off_start, off_end) in enumerate(offsets):
      if off_start == off_end: continue
      if off_end > char_start and off_start < char_end:
        if IS_SCALE:
          scale_map[0, t_i] -= 10
        break
  else:
    return "⚠️ Gold key not found in the JSON section of the prompt."

  model.config.current_scale_map = scale_map

  # ここをmodel.generateにするか、calculate_nllにするか、良くない構造だけど
  nll_dict, attention_map = calculate_nll(model, tokenizer, prompt, value, kv_char_positions)

  return nll_dict, attention_map

def get_records_at_position(records, sample_num, target_key, target_value, pos):
    gold_pair = [target_key, target_value]
    other_records = [r for r in records if r != gold_pair]
    # 他のレコードの順序は固定し、指定された数だけ抽出
    selected_others = other_records[:sample_num - 1]

    # 指定された位置に正解を挿入
    selected_others.insert(pos, gold_pair)
    return selected_others


In [7]:
# 設定
KV_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/KV/kv-retrieval-300_keys.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/kv_retrieval/attnmap_kv_retrieval_no_scale.jsonl"

KV_PAIR_NUM = 50
SAMPLR_NUM = 100
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

IS_SCALE = False

import torch
import torch.nn as nn
import traceback
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def run_kv_retrieval(records, key, value, model, tokenizer):
    # 1. プロンプトの作成
    prompt, context_end_idx, kv_char_positions = format_kv_prompt(records, key)
    full_text = prompt + " " + value

    # 2. トークナイズ
    inputs = tokenizer(full_text, return_tensors="pt", return_offsets_mapping=True).to(model.device)
    input_ids = inputs.input_ids
    offsets = inputs.offset_mapping[0]
    seq_len = input_ids.shape[1]

    # ターゲット（正解）の長さを取得
    target_ids = tokenizer(value, add_special_tokens=False).input_ids
    target_len = len(target_ids)

    # 3. スケールマップの作成
    char_start = prompt.find(key, 0, context_end_idx)
    char_end = char_start + len(key)
    scale_map = torch.full((1, seq_len), 1.0, device=model.device, dtype=torch.float16)

    found_gold = False
    if char_start != -1:
        for t_i, (off_start, off_end) in enumerate(offsets):
            if off_start == off_end: continue
            if off_end > char_start and off_start < char_end:
                scale_map[0, t_i] -= 10 # 実験用のスケーリング
                found_gold = True
                break

    if not found_gold:
        return "⚠️ Gold key not found"

    model.config.current_scale_map = scale_map

    # --- 4. Hook（フック）の設定：15-20層の最後1トークンだけを抜き出す ---
    extracted_attn = {}
    target_layer_indices = range(14, 20) # 15層(index 14)から20層(index 19)

    def hook_fn(layer_idx):
        def fn(module, input, output):
          if hasattr(module, 'last_q_scaled') and hasattr(module, 'last_k_scaled'):
            q = module.last_q_scaled
            k = module.last_k_scaled

            d_k = q.size(-1)
            scores = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)

            attn_weights = torch.nn.functional.softmax(scores, dim=-1)
            extracted_attn[layer_idx] = attn_weights
        return fn

    hooks = []
    for i in target_layer_indices:
        # モデルの構造に合わせてパスを指定。Llama3などは model.layers[i].self_attn
        layer = model.model.layers[i].self_attn
        hooks.append(layer.register_forward_hook(hook_fn(i)))

    # 5. 推論実行
    try:
        with torch.no_grad():
            # output_attentions=Trueにするとメモリが爆発するので、FalseのままHookで抜く
            outputs = model(input_ids, output_attentions=False)

        # 6. NLL（損失）の計算
        logits = outputs.logits
        target_logits = logits[0, -(target_len + 1):-1, :]
        target_labels = input_ids[0, -target_len:]

        loss_fct = nn.CrossEntropyLoss(reduction="mean")
        avg_nll = loss_fct(target_logits, target_labels).item()

        # 7. アテンションマップの整理 (Layers, Batch, Heads, 1, SeqLen)
        # すべてのターゲット層が取得できているか確認してスタック
        final_attn_tensor = torch.stack([extracted_attn[i] for i in target_layer_indices])

        # トークン位置の変換（文字レベル -> トークンレベル）
        kv_token_positions = []
        for s_char, e_char in kv_char_positions:
            s_token = inputs.char_to_token(s_char)
            e_token = inputs.char_to_token(e_char - 1)
            kv_token_positions.append((s_token, e_token))

        return {
            "avg_nll": avg_nll,
            "target_len": target_len,
            "last_token_attentions": final_attn_tensor.numpy().tolist(), # numpyならJSON保存時に .tolist() してね
            "kv_token_positions": kv_token_positions,
        }

    finally:
        # メモリリーク防止のため必ずフックを解除
        for h in hooks:
            h.remove()

def main():
  model, tokenizer = get_model_safe(MODEL_NAME)

  with xopen(KV_PATH, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
    for line in tqdm(fin, desc="Running KV Retrieval..."):
      data = json.loads(line)
      key = data["key"]
      value = data["value"]
      records = data["ordered_kv_records"]

      for i in range(KV_PAIR_NUM)[:1]:
        records = get_records_at_position(records, KV_PAIR_NUM, key, value, i)

        try:
          nll_dict= run_kv_retrieval(records, key, value, model, tokenizer)
        except Exception:
            print("\n" + "="*50)
            print("🚨 Exception occurred! Detailed Traceback:")
            # ここでスタックトレースを表示
            traceback.print_exc()
            print("="*50 + "\n")
            continue

        result_record = {
            **nll_dict,
            "key": key,
            "value": value,
            "num_kv": len(records),
        }

        fout.write(json.dumps(result_record)+"\n")
        fout.flush()

if __name__ == "__main__":
  main()

✅ Model already loaded. Using cached model.


Running KV Retrieval...: 0it [00:00, ?it/s]


TypeError: 'tqdm' object is not subscriptable

In [ ]:
import torch
import gc

def flush():
    # 1. 巨大なテンソルを削除（もし変数名がわかっている場合）
    # del model, outputs, last_token_attentions

    # 2. Pythonのガベージコレクションを強制実行
    gc.collect()

    # 3. PyTorchのキャッシュ（未使用だが確保されているVRAM）を解放
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

flush()

In [3]:
import torch

def test_hook_functionality(model, tokenizer):
    print("🧪 Starting Hook Functionality Test...")

    # データを格納するグローバル辞書（テスト用）
    test_captured_results = {}

    # 1. テスト用のHook関数
    def test_extraction_hook(layer_idx):
        def hook(module, input, output):
            # 自作クラスの属性にアクセス
            if hasattr(module, 'last_q_scaled') and hasattr(module, 'last_k_scaled'):
                q = module.last_q_scaled
                k = module.last_k_scaled

                # スコアを計算 (Q @ K.T)
                d_k = q.size(-1)
                scores = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)
                weights = torch.nn.functional.softmax(scores, dim=-1)

                test_captured_results[layer_idx] = {
                    "q_shape": q.shape,
                    "k_shape": k.shape,
                    "weights_shape": weights.shape,
                    "sample_weights": weights[0, 0, 0, :5] # 最初のヘッドの先頭5要素
                }
            else:
                print(f"   ⚠️ Warning: layer {layer_idx} does not have last_q_scaled attribute.")
        return hook

    # debug用のhook
    def debug_access_hook(layer_idx):
      def hook(module, input, output):
          # 1. クラス名を確認
          class_name = type(module).__name__

          # 2. クラス内の変数（selfに相当するもの）にアクセス
          # layer_idx は初期化時に self.layer_idx = layer_idx としているはず
          l_idx = getattr(module, 'layer_idx', "N/A")
          h_dim = getattr(module, 'head_dim', "N/A")

          print(f"🔍 [Hook Active] Class: {class_name} | Layer: {l_idx} | Head Dim: {h_dim}")

          # 3. 前回の失敗原因（last_q_scaledがあるか）を直接チェック
          has_q = hasattr(module, 'last_q_scaled')
          print(f"   -> has 'last_q_scaled': {has_q}")

      return hook

    # 2. Hookの登録 (パッチを当てた15-20層が対象)
    target_layers = list(range(14, 20))
    handles = []
    for i in target_layers:
        layer = model.model.layers[i].self_attn
        handle = layer.register_forward_hook(test_extraction_hook(i))
        handles.append(handle)

    # 3. ダミー推論の実行
    input_text = "This is a hook test for KV retrieval."
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    print(f"🏃 Running inference with input length: {inputs.input_ids.shape[1]}...")

    try:
        with torch.no_grad():
            # output_attentions=False でOKなことを確認
            model(**inputs, output_attentions=False)

        # 4. 結果の検証
        print("\n📊 Hook Results:")
        if not test_captured_results:
            print("   ❌ FAILED: No data captured in test_captured_results.")
        else:
            for layer_idx, data in test_captured_results.items():
                print(f"   ✅ Layer {layer_idx}:")
                print(f"      - Q shape: {data['q_shape']}") # (bsz, heads, 1, head_dim)
                print(f"      - K shape: {data['k_shape']}") # (bsz, heads, seq_len, head_dim)
                print(f"      - Weights shape: {data['weights_shape']}") # (bsz, heads, 1, seq_len)
                print(f"      - Sample weights (first 5): {data['sample_weights'].tolist()}")

    finally:
        # 5. 後片付け
        for h in handles:
            h.remove()
        print("\n🧹 Hooks removed.")

# 実行例
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
CACHED_MODEL, CACHED_TOKENIZER = get_model_safe(MODEL_NAME)
test_hook_functionality(CACHED_MODEL, CACHED_TOKENIZER)


⏳ Loading new model... (This may take a while)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.
🧪 Starting Hook Functionality Test...
🏃 Running inference with input length: 12...

📊 Hook Results:
   ✅ Layer 14:
      - Q shape: torch.Size([1, 32, 1, 128])
      - K shape: torch.Size([1, 32, 12, 128])
      - Weights shape: torch.Size([1, 32, 1, 12])
      - Sample weights (first 5): [0.266845703125, 0.06817626953125, 0.0504150390625, 0.049591064453125, 0.05303955078125]
   ✅ Layer 15:
      - Q shape: torch.Size([1, 32, 1, 128])
      - K shape: torch.Size([1, 32, 12, 128])
      - Weights shape: torch.Size([1, 32, 1, 12])
      - Sample weights (first 5): [0.41943359375, 0.065185546875, 0.01482391357421875, 0.00997161865234375, 0.01177215576171875]
   ✅ Layer 16:
      - Q shape: torch.Size([1, 32, 1, 128])
      - K shape: torch.Size([1, 32, 12, 128])
      - Weights shape: torch.Size([1, 32, 1, 12])
      - Sample weights (first 5): [0.481201171875, 0.00809478759765625, 0.0

TypeError: LlamaAttention.forward() missing 2 required positional arguments: 'position_embeddings' and 'attention_mask'